# Imports

In [1]:
import os

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr, pointbiserialr
from scipy.stats import f_oneway, kruskal

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, cohen_kappa_score, make_scorer

from autogluon.tabular import TabularDataset, TabularPredictor
#from autogluon.multimodal import MultiModalPredictor
import xgboost as xgb

plt.style.use('default')
sns.set_palette("husl")
SEED = 43
np.random.seed(SEED)

SCORING = {
    'roc_auc': 'roc_auc',
    'accuracy': 'accuracy',
    'f1': 'f1',
    'kappa': make_scorer(cohen_kappa_score),
}
GRID_SEARCH_KWARGS = dict(
    cv=10,
    scoring=SCORING,
    refit='kappa',
    n_jobs=-1,
    return_train_score=True
)
LINEAR_PARAM_GRID = {
    'model__C': [0.1, 1.0, 10.0],
    'model__penalty': ['l2'],
    'model__solver': ['lbfgs'],
    'model__max_iter': [1000],
}
TREE_PARAM_GRID = {
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [2, 100],
    'model__min_samples_leaf': [2, 100],
    'model__max_features': [None, 'sqrt', 'log2']
}
RF_PARAM_GRID = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [2, 100],
    'model__min_samples_leaf': [2, 100],
    'model__max_features': [None, 'sqrt', 'log2']
}
MLP_PARAM_GRID = {
    'model__hidden_layer_sizes': [(100,), (100, 50), (50, 25)],
    'model__activation': ['relu', 'tanh'],
    'model__alpha': [0.0001, 0.001, 0.01],
    'model__learning_rate_init': [0.001, 0.01]
}
XGB_PARAM_GRID = {
    'model__learning_rate': [0.01, 0.1, 0.2],
    'model__n_estimators': [50, 100, 200]
}

def evaluate_classification_metrics(y_true, y_pred, y_score=None):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    if y_score is None:
        y_score = y_pred

    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
    }
    if len(np.unique(y_true)) > 1:
        metrics['roc_auc'] = roc_auc_score(y_true, y_score)
        metrics['kappa'] = cohen_kappa_score(y_true, y_pred)
    else:
        metrics['roc_auc'] = float('nan')
        metrics['kappa'] = float('nan')
    return metrics


def record_classification_results(results, target, model_name, y_true, y_pred, y_score=None, split='test'):
    metrics = evaluate_classification_metrics(y_true, y_pred, y_score=y_score)
    split_label = split.capitalize()
    print(f"{split_label} ROC-AUC {metrics['roc_auc']:.4f}")
    print(f"{split_label} Accuracy {metrics['accuracy']:.4f}")
    print(f"{split_label} F1 {metrics['f1']:.4f}")
    print(f"{split_label} Kappa {metrics['kappa']:.4f}")
    results.append({
        'target': target,
        'model': model_name,
        f'{split}_roc_auc': metrics['roc_auc'],
        f'{split}_accuracy': metrics['accuracy'],
        f'{split}_f1': metrics['f1'],
        f'{split}_kappa': metrics['kappa'],
    })
    return metrics


# Load data

In [3]:
data_path_file = "yolo_llm-metafeatures_disc.csv"
data = pd.read_csv("../data/" + data_path_file, index_col=0)

print(data)

      Time_of_Day Weather_Condition Road_Environment_Type  \
52       Twilight            Cloudy                 Urban   
71       Daylight            Cloudy                 Urban   
128     Dawn/Dusk            Cloudy                 Urban   
256      Daylight            Cloudy                 Urban   
621      Daylight            Cloudy                 Urban   
...           ...               ...                   ...   
99783       Night            Cloudy                 Urban   
99862       Night             Clear                 Urban   
99885    Daylight             Clear                 Rural   
99929    Daylight             Clear                 Rural   
99936    Daylight             Clear                 Rural   

      Road_Surface_Condition Traffic_Density Pedestrian_Presence  \
52                       Dry             Low                 NaN   
71                       Dry        Moderate         On_Sidewalk   
128                      Dry             Low         On_Sidewal

In [4]:
#data = data.dropna().reset_index(drop=True)
data = data.fillna("None")
num_rows, num_cols = data.shape
print(f"After dropping missing values: {num_rows:,} rows and {num_cols} columns")

After dropping missing values: 1,010 rows and 24 columns


In [5]:
iou = data["iou"]
lrp = data["lrp"]
conf = data["conf"]
image_paths_dict = {int(img_pth.split("_")[0]):f"../data/zod_yolo/images/val/{img_pth}" for img_pth in os.listdir("../data/zod_yolo/images/val") if img_pth.endswith(".jpg")}
img_path = pd.DataFrame.from_dict(image_paths_dict, orient='index', columns=["Images"])

data = data.drop(columns=["iou", "lrp"])
data_indices = data.index.to_numpy()

In [6]:
all_columns = data.columns    
#all_columns = all_columns.drop(["weather_code", "cloud_cover_low", "cloud_cover_mid"])

print(f"Number of rows: {data.shape[0]}")
print(f"Columns: {all_columns.sort_values()}")
print(f"Number of columns: {len(all_columns)}")

Number of rows: 1010
Columns: Index(['Artificial_Lighting_Source', 'Building_Density', 'Crosswalk_Presence',
       'Cyclist_Presence', 'Ego_Vehicle_Lane_POSITION',
       'Ego_Vehicle_Lane_Position', 'Horizontal_Road_Curvature',
       'Infrastructure_Barriers', 'Intersection_Type',
       'Lane_Marking_Visibility', 'Number_of_Visible_Lanes',
       'Pedestrian_Presence', 'Road_Environment_Type',
       'Road_Surface_Condition', 'Time_of_Day', 'Traffic_Density',
       'Traffic_Sign_Presence', 'Vegetation_Level', 'Vehicle_Composition',
       'Vertical_Road_Profile', 'Weather_Condition', 'conf'],
      dtype='object')
Number of columns: 22


In [ ]:
# Metafeatures
numeric_columns = ['conf', 'solar_angle_elevation', 'temperature_2m', 'relative_humidity_2m', 'rain', 'snowfall', 'cloud_cover', 'wind_speed_10m', 'forward_acceleration', 'lateral_acceleration', 'forward_velocity', 'lateral_velocity', 'field_view_horizontal', 'camera_distance_from_ground', 'camera_pitch_angle', 'distortion_magnitude', 'camera_offset', 'laplacian', 'brightness', 'contrast', 'sharpness', 'noisiness', 'quality', 'complexity']

categorical_columns = ["country", "time_of_day", "road_type", "road_condition", "weather"]
other = ["hour", "month", "lat", "long"]
print(sorted(numeric_columns + categorical_columns + other))
assert len(all_columns) == (len(categorical_columns) + len(numeric_columns) + len(other)), "Columns not match"

['brightness', 'camera_distance_from_ground', 'camera_offset', 'camera_pitch_angle', 'cloud_cover', 'complexity', 'conf', 'contrast', 'country', 'distortion_magnitude', 'field_view_horizontal', 'forward_acceleration', 'forward_velocity', 'hour', 'laplacian', 'lat', 'lateral_acceleration', 'lateral_velocity', 'long', 'month', 'noisiness', 'quality', 'rain', 'relative_humidity_2m', 'road_condition', 'road_type', 'sharpness', 'snowfall', 'solar_angle_elevation', 'temperature_2m', 'time_of_day', 'weather', 'wind_speed_10m']


AssertionError: Columns not match

In [7]:
# llm-metafeatures
categorical_columns = ['Number_of_Visible_Lanes', 'Artificial_Lighting_Source', 'Building_Density', 'Crosswalk_Presence', 'Cyclist_Presence', 'Ego_Vehicle_Lane_POSITION','Ego_Vehicle_Lane_Position', 'Horizontal_Road_Curvature','Infrastructure_Barriers', 'Intersection_Type','Lane_Marking_Visibility', 'Pedestrian_Presence', 'Road_Environment_Type','Road_Surface_Condition', 'Time_of_Day', 'Traffic_Density','Traffic_Sign_Presence', 'Vegetation_Level', 'Vehicle_Composition','Vertical_Road_Profile', 'Weather_Condition']

numeric_columns = ["conf"]
print(sorted(numeric_columns + categorical_columns))
assert len(all_columns) == (len(categorical_columns) + len(numeric_columns)), "Columns not match"

['Artificial_Lighting_Source', 'Building_Density', 'Crosswalk_Presence', 'Cyclist_Presence', 'Ego_Vehicle_Lane_POSITION', 'Ego_Vehicle_Lane_Position', 'Horizontal_Road_Curvature', 'Infrastructure_Barriers', 'Intersection_Type', 'Lane_Marking_Visibility', 'Number_of_Visible_Lanes', 'Pedestrian_Presence', 'Road_Environment_Type', 'Road_Surface_Condition', 'Time_of_Day', 'Traffic_Density', 'Traffic_Sign_Presence', 'Vegetation_Level', 'Vehicle_Composition', 'Vertical_Road_Profile', 'Weather_Condition', 'conf']


# Assessor Tests

Split data into train 60% and validation 40%

In [8]:
#X_train, X_test, y_train, y_test = train_test_split(data, test_size=0.4, train_size=0.6, stratify=data["iou"])
train_idx, test_idx = train_test_split(data_indices, test_size=0.4, train_size=0.6)

X_train = data.loc[train_idx]
y_train = iou.loc[train_idx]
y_train_lrp = lrp.loc[train_idx]
conf_train = conf.loc[train_idx]
conf_train = pd.DataFrame(conf_train)

X_test = data.loc[test_idx]
y_test = iou.loc[test_idx]
y_test_lrp = lrp.loc[test_idx]
conf_test = conf.loc[test_idx]
conf_test = pd.DataFrame(conf_test)

print("X:", len(X_train), len(X_test))
print("y:", len(y_train), len(y_test))
print("Conf:", len(conf_train), len(conf_test))
print(X_train.columns)
CLASS_THRESHOLD = 0.5

y_train_cls = (y_train >= CLASS_THRESHOLD).astype(int)
y_test_cls = (y_test >= CLASS_THRESHOLD).astype(int)
y_train_lrp_cls = (y_train_lrp >= CLASS_THRESHOLD).astype(int)
y_test_lrp_cls = (y_test_lrp >= CLASS_THRESHOLD).astype(int)


X: 606 404
y: 606 404
Conf: 606 404
Index(['Time_of_Day', 'Weather_Condition', 'Road_Environment_Type',
       'Road_Surface_Condition', 'Traffic_Density', 'Pedestrian_Presence',
       'Cyclist_Presence', 'Number_of_Visible_Lanes', 'Intersection_Type',
       'Lane_Marking_Visibility', 'Artificial_Lighting_Source',
       'Crosswalk_Presence', 'Vertical_Road_Profile',
       'Horizontal_Road_Curvature', 'Infrastructure_Barriers',
       'Building_Density', 'Vegetation_Level', 'Traffic_Sign_Presence',
       'Vehicle_Composition', 'Ego_Vehicle_Lane_Position',
       'Ego_Vehicle_Lane_POSITION', 'conf'],
      dtype='object')


In [9]:
model_results = []

### IoU

#### Baseline

Random Prediction. Mean of metric over training set.

In [10]:
mean_iou_train = np.mean(y_train_cls)
iou_pred_test = np.full_like(y_test_cls, int(mean_iou_train >= 0.5))
record_classification_results(model_results, 'IoU', 'Constant Mean Predictor', y_test_cls, iou_pred_test)


Test ROC-AUC 0.5000
Test Accuracy 0.9926
Test F1 0.0000
Test Kappa 0.0000


{'accuracy': 0.9925742574257426,
 'precision': 0.0,
 'recall': 0.0,
 'f1': 0.0,
 'roc_auc': 0.5,
 'kappa': 0.0}

#### Linear Reg on Conf

In [11]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['conf']),
    ],
    remainder='passthrough'
)
linear_reg_conf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression())
])
linear_reg_conf_grid = GridSearchCV(
    linear_reg_conf_pipeline,
    param_grid=LINEAR_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
linear_reg_conf_grid.fit(conf_train, y_train_cls)

best_idx = linear_reg_conf_grid.best_index_
mean_train_roc_auc = linear_reg_conf_grid.cv_results_['mean_train_roc_auc'][best_idx]
mean_test_roc_auc = linear_reg_conf_grid.cv_results_['mean_test_roc_auc'][best_idx]
mean_train_accuracy = -linear_reg_conf_grid.cv_results_['mean_train_accuracy'][best_idx]
mean_test_accuracy = -linear_reg_conf_grid.cv_results_['mean_test_accuracy'][best_idx]
mean_train_f1 = -linear_reg_conf_grid.cv_results_['mean_train_f1'][best_idx]
mean_test_f1 = -linear_reg_conf_grid.cv_results_['mean_test_f1'][best_idx]
print(f"Mean CV train ROC-AUC {mean_train_roc_auc:.4f}")
print(f"Mean CV test ROC-AUC {mean_test_roc_auc:.4f}")
print(f"Mean CV train Accuracy {mean_train_accuracy:.4f}")
print(f"Mean CV test Accuracy {mean_test_accuracy:.4f}")
print(f"Mean CV train F1 {mean_train_f1:.4f}")
print(f"Mean CV test F1 {mean_test_f1:.4f}")
print(f"Best params: {linear_reg_conf_grid.best_params_}")
best_linear_reg_conf = linear_reg_conf_grid.best_estimator_


Mean CV train ROC-AUC 0.9408
Mean CV test ROC-AUC 0.9494
Mean CV train Accuracy -0.9802
Mean CV test Accuracy -0.9786
Mean CV train F1 -0.1423
Mean CV test F1 -0.1000
Best params: {'model__C': 1.0, 'model__max_iter': 1000, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}


In [12]:
y_pred_iou_baseline = best_linear_reg_conf.predict(conf_test)
record_classification_results(model_results, 'IoU', 'Univariate Linear Regression (Confidence)', y_test_cls, y_pred_iou_baseline)


Test ROC-AUC 0.4963
Test Accuracy 0.9851
Test F1 0.0000
Test Kappa -0.0075


{'accuracy': 0.9851485148514851,
 'precision': 0.0,
 'recall': 0.0,
 'f1': 0.0,
 'roc_auc': 0.49625935162094764,
 'kappa': -0.007481296758104605}

#### Linear Regression

Train models with cv and then test.

In [13]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
linear_reg_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression())
])
linear_reg_grid = GridSearchCV(
    linear_reg_pipeline,
    param_grid=LINEAR_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
linear_reg_grid.fit(X_train, y_train_cls)

best_idx = linear_reg_grid.best_index_
mean_train_roc_auc = linear_reg_grid.cv_results_['mean_train_roc_auc'][best_idx]
mean_test_roc_auc = linear_reg_grid.cv_results_['mean_test_roc_auc'][best_idx]
mean_train_accuracy = -linear_reg_grid.cv_results_['mean_train_accuracy'][best_idx]
mean_test_accuracy = -linear_reg_grid.cv_results_['mean_test_accuracy'][best_idx]
mean_train_f1 = -linear_reg_grid.cv_results_['mean_train_f1'][best_idx]
mean_test_f1 = -linear_reg_grid.cv_results_['mean_test_f1'][best_idx]
print(f"Mean CV train ROC-AUC {mean_train_roc_auc:.4f}")
print(f"Mean CV test ROC-AUC {mean_test_roc_auc:.4f}")
print(f"Mean CV train Accuracy {mean_train_accuracy:.4f}")
print(f"Mean CV test Accuracy {mean_test_accuracy:.4f}")
print(f"Mean CV train F1 {mean_train_f1:.4f}")
print(f"Mean CV test F1 {mean_test_f1:.4f}")
print(f"Best params: {linear_reg_grid.best_params_}")
best_linear_reg = linear_reg_grid.best_estimator_

/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [5, 6] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [5, 6] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [5, 6] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.p

Mean CV train ROC-AUC 0.9735
Mean CV test ROC-AUC 0.9451
Mean CV train Accuracy -0.9785
Mean CV test Accuracy -0.9786
Mean CV train F1 -0.0000
Mean CV test F1 -0.0000
Best params: {'model__C': 0.1, 'model__max_iter': 1000, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}


/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [13] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: U

In [14]:
y_pred_iou_lr = best_linear_reg.predict(X_test)
record_classification_results(model_results, 'IoU', 'Linear Regression', y_test_cls, y_pred_iou_lr)


Test ROC-AUC 0.5000
Test Accuracy 0.9926
Test F1 0.0000
Test Kappa 0.0000


{'accuracy': 0.9925742574257426,
 'precision': 0.0,
 'recall': 0.0,
 'f1': 0.0,
 'roc_auc': 0.5,
 'kappa': 0.0}

#### Decision Trees

In [15]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
decision_tree_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeClassifier(random_state=SEED))
])
decision_tree_grid = GridSearchCV(
    decision_tree_pipeline,
    param_grid=TREE_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
decision_tree_grid.fit(X_train, y_train_cls)

best_idx = decision_tree_grid.best_index_
mean_train_roc_auc = decision_tree_grid.cv_results_['mean_train_roc_auc'][best_idx]
mean_test_roc_auc = decision_tree_grid.cv_results_['mean_test_roc_auc'][best_idx]
mean_train_accuracy = -decision_tree_grid.cv_results_['mean_train_accuracy'][best_idx]
mean_test_accuracy = -decision_tree_grid.cv_results_['mean_test_accuracy'][best_idx]
mean_train_f1 = -decision_tree_grid.cv_results_['mean_train_f1'][best_idx]
mean_test_f1 = -decision_tree_grid.cv_results_['mean_test_f1'][best_idx]
print(f"Mean CV train ROC-AUC {mean_train_roc_auc:.4f}")
print(f"Mean CV test ROC-AUC {mean_test_roc_auc:.4f}")
print(f"Mean CV train Accuracy {mean_train_accuracy:.4f}")
print(f"Mean CV test Accuracy {mean_test_accuracy:.4f}")
print(f"Mean CV train F1 {mean_train_f1:.4f}")
print(f"Mean CV test F1 {mean_test_f1:.4f}")
print(f"Best params: {decision_tree_grid.best_params_}")
best_decision_tree = decision_tree_grid.best_estimator_


/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [5, 6] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [5, 6] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:2

Mean CV train ROC-AUC 0.9173
Mean CV test ROC-AUC 0.9164
Mean CV train Accuracy -0.9785
Mean CV test Accuracy -0.9786
Mean CV train F1 -0.0000
Mean CV test F1 -0.0000
Best params: {'model__max_depth': None, 'model__max_features': None, 'model__min_samples_leaf': 100, 'model__min_samples_split': 2}


In [16]:
y_pred_iou_dt = best_decision_tree.predict(X_test)
record_classification_results(model_results, 'IoU', 'Decision Tree', y_test_cls, y_pred_iou_dt)


Test ROC-AUC 0.5000
Test Accuracy 0.9926
Test F1 0.0000
Test Kappa 0.0000


{'accuracy': 0.9925742574257426,
 'precision': 0.0,
 'recall': 0.0,
 'f1': 0.0,
 'roc_auc': 0.5,
 'kappa': 0.0}

#### Random Forest

In [17]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
rand_forest_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(random_state=SEED, n_jobs=-1))
])
rand_forest_grid = GridSearchCV(
    rand_forest_pipeline,
    param_grid=RF_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
rand_forest_grid.fit(X_train, y_train_cls)

best_idx = rand_forest_grid.best_index_
mean_train_roc_auc = rand_forest_grid.cv_results_['mean_train_roc_auc'][best_idx]
mean_test_roc_auc = rand_forest_grid.cv_results_['mean_test_roc_auc'][best_idx]
mean_train_accuracy = -rand_forest_grid.cv_results_['mean_train_accuracy'][best_idx]
mean_test_accuracy = -rand_forest_grid.cv_results_['mean_test_accuracy'][best_idx]
mean_train_f1 = -rand_forest_grid.cv_results_['mean_train_f1'][best_idx]
mean_test_f1 = -rand_forest_grid.cv_results_['mean_test_f1'][best_idx]
print(f"Mean CV train ROC-AUC {mean_train_roc_auc:.4f}")
print(f"Mean CV test ROC-AUC {mean_test_roc_auc:.4f}")
print(f"Mean CV train Accuracy {mean_train_accuracy:.4f}")
print(f"Mean CV test Accuracy {mean_test_accuracy:.4f}")
print(f"Mean CV train F1 {mean_train_f1:.4f}")
print(f"Mean CV test F1 {mean_test_f1:.4f}")
print(f"Best params: {rand_forest_grid.best_params_}")
best_rand_forest = rand_forest_grid.best_estimator_


/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [13] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [5, 6] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246

Mean CV train ROC-AUC 0.9998
Mean CV test ROC-AUC 0.9122
Mean CV train Accuracy -0.9899
Mean CV test Accuracy -0.9786
Mean CV train F1 -0.6811
Mean CV test F1 -0.0000
Best params: {'model__max_depth': None, 'model__max_features': None, 'model__min_samples_leaf': 2, 'model__min_samples_split': 2, 'model__n_estimators': 100}


In [18]:
y_pred_iou_rf = best_rand_forest.predict(X_test)
record_classification_results(model_results, 'IoU', 'Random Forest', y_test_cls, y_pred_iou_rf)


Test ROC-AUC 0.4975
Test Accuracy 0.9876
Test F1 0.0000
Test Kappa -0.0060


{'accuracy': 0.9876237623762376,
 'precision': 0.0,
 'recall': 0.0,
 'f1': 0.0,
 'roc_auc': 0.4975062344139651,
 'kappa': -0.005976095617529875}

#### MLP

In [19]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
mlp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', MLPClassifier(max_iter=500, random_state=SEED))
])
mlp_grid = GridSearchCV(
    mlp_pipeline,
    param_grid=MLP_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
mlp_grid.fit(X_train, y_train_cls)

best_idx = mlp_grid.best_index_
mean_train_roc_auc = mlp_grid.cv_results_['mean_train_roc_auc'][best_idx]
mean_test_roc_auc = mlp_grid.cv_results_['mean_test_roc_auc'][best_idx]
mean_train_accuracy = -mlp_grid.cv_results_['mean_train_accuracy'][best_idx]
mean_test_accuracy = -mlp_grid.cv_results_['mean_test_accuracy'][best_idx]
mean_train_f1 = -mlp_grid.cv_results_['mean_train_f1'][best_idx]
mean_test_f1 = -mlp_grid.cv_results_['mean_test_f1'][best_idx]
print(f"Mean CV train ROC-AUC {mean_train_roc_auc:.4f}")
print(f"Mean CV test ROC-AUC {mean_test_roc_auc:.4f}")
print(f"Mean CV train Accuracy {mean_train_accuracy:.4f}")
print(f"Mean CV test Accuracy {mean_test_accuracy:.4f}")
print(f"Mean CV train F1 {mean_train_f1:.4f}")
print(f"Mean CV test F1 {mean_test_f1:.4f}")
print(f"Best params: {mlp_grid.best_params_}")
best_mlp = mlp_grid.best_estimator_


/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [13] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [13] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [5, 6] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:24

Mean CV train ROC-AUC 1.0000
Mean CV test ROC-AUC 0.9162
Mean CV train Accuracy -0.9994
Mean CV test Accuracy -0.9670
Mean CV train F1 -0.9877
Mean CV test F1 -0.1000
Best params: {'model__activation': 'relu', 'model__alpha': 0.001, 'model__hidden_layer_sizes': (100, 50), 'model__learning_rate_init': 0.01}


In [20]:
y_pred_iou_mlp = best_mlp.predict(X_test)
record_classification_results(model_results, 'IoU', 'MLP', y_test_cls, y_pred_iou_mlp)


Test ROC-AUC 0.6592
Test Accuracy 0.9802
Test F1 0.2000
Test Kappa 0.1916


{'accuracy': 0.9801980198019802,
 'precision': 0.14285714285714285,
 'recall': 0.3333333333333333,
 'f1': 0.2,
 'roc_auc': 0.6591853699085619,
 'kappa': 0.19159579789894954}

#### XGBoost

In [21]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb.XGBClassifier(eval_metric="logloss"))
])
xgb_grid = GridSearchCV(
    xgb_pipeline,
    param_grid=XGB_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
xgb_grid.fit(X_train, y_train_cls)


best_idx = xgb_grid.best_index_
mean_train_roc_auc = xgb_grid.cv_results_['mean_train_roc_auc'][best_idx]
mean_test_roc_auc = xgb_grid.cv_results_['mean_test_roc_auc'][best_idx]
mean_train_accuracy = -xgb_grid.cv_results_['mean_train_accuracy'][best_idx]
mean_test_accuracy = -xgb_grid.cv_results_['mean_test_accuracy'][best_idx]
mean_train_f1 = -xgb_grid.cv_results_['mean_train_f1'][best_idx]
mean_test_f1 = -xgb_grid.cv_results_['mean_test_f1'][best_idx]
print(f"Mean CV train ROC-AUC {mean_train_roc_auc:.4f}")
print(f"Mean CV test ROC-AUC {mean_test_roc_auc:.4f}")
print(f"Mean CV train Accuracy {mean_train_accuracy:.4f}")
print(f"Mean CV test Accuracy {mean_test_accuracy:.4f}")
print(f"Mean CV train F1 {mean_train_f1:.4f}")
print(f"Mean CV test F1 {mean_test_f1:.4f}")
print(f"Best params: {xgb_grid.best_params_}")
best_xgb = xgb_grid.best_estimator_

Mean CV train ROC-AUC 0.9999
Mean CV test ROC-AUC 0.9316
Mean CV train Accuracy -0.9980
Mean CV test Accuracy -0.9654
Mean CV train F1 -0.9492
Mean CV test F1 -0.0667
Best params: {'model__learning_rate': 0.1, 'model__n_estimators': 100}


In [22]:
y_pred_iou_xgb = best_xgb.predict(X_test)
record_classification_results(model_results, 'IoU', 'XGBoost', y_test_cls, y_pred_iou_xgb)


Test ROC-AUC 0.4963
Test Accuracy 0.9851
Test F1 0.0000
Test Kappa -0.0075


{'accuracy': 0.9851485148514851,
 'precision': 0.0,
 'recall': 0.0,
 'f1': 0.0,
 'roc_auc': 0.49625935162094764,
 'kappa': -0.007481296758104605}

#### Autogluon

##### Tabluar

In [23]:
train = pd.merge(X_train, y_train_cls, left_index=True, right_index=True)

In [24]:
autoglue_model = TabularPredictor(label="iou", problem_type="binary", eval_metric="roc_auc", path="../models/assessors/autoglue_tab/iou/")
autoglue_predictor = autoglue_model.fit(train)


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.14
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Debian 6.12.48-1 (2025-09-20)
CPU Count:          32
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 23.56/23.56 GB
Total GPU Memory:   Free: 23.56 GB, Allocated: 0.00 GB, Total: 23.56 GB
GPU Count:          1
Memory Avail:       6.12 GB / 30.47 GB (20.1%)
Disk Space Avail:   1389.81 GB / 3542.28 GB (39.2%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : New in v1.5: The state-of-the-art for tabular data. Massively better than 'best' on datasets <100000 samples by using new Tabular 

In [25]:
autoglue_predictor.fit_summary(verbosity=2)

*** Summary of fit() ***
Estimated performance of each model:
                  model  score_val eval_metric  pred_time_val  fit_time  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0   WeightedEnsemble_L2   0.969188     roc_auc       0.015207  2.094349                0.000181           0.013683            2       True         12
1            LightGBMXT   0.941176     roc_auc       0.002764  0.257893                0.002764           0.257893            1       True          1
2              CatBoost   0.913165     roc_auc       0.003005  1.215708                0.003005           1.215708            1       True          5
3        NeuralNetTorch   0.910364     roc_auc       0.006945  1.387711                0.006945           1.387711            1       True         10
4        ExtraTreesGini   0.890756     roc_auc       0.025995  0.225461                0.025995           0.225461            1       True          6
5      RandomForestGini   0.879552    

/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/autogluon/core/utils/plots.py:169: UserWarning: AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"
  warnings.warn('AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"')


{'model_types': {'LightGBMXT': 'LGBModel',
  'LightGBM': 'LGBModel',
  'RandomForestGini': 'RFModel',
  'RandomForestEntr': 'RFModel',
  'CatBoost': 'CatBoostModel',
  'ExtraTreesGini': 'XTModel',
  'ExtraTreesEntr': 'XTModel',
  'NeuralNetFastAI': 'NNFastAiTabularModel',
  'XGBoost': 'XGBoostModel',
  'NeuralNetTorch': 'TabularNeuralNetTorchModel',
  'LightGBMLarge': 'LGBModel',
  'WeightedEnsemble_L2': 'WeightedEnsembleModel'},
 'model_performance': {'LightGBMXT': 0.9411764705882353,
  'LightGBM': 0.7591036414565826,
  'RandomForestGini': 0.8795518207282913,
  'RandomForestEntr': 0.876750700280112,
  'CatBoost': 0.9131652661064426,
  'ExtraTreesGini': 0.8907563025210085,
  'ExtraTreesEntr': 0.8361344537815125,
  'NeuralNetFastAI': 0.7983193277310924,
  'XGBoost': 0.7703081232492996,
  'NeuralNetTorch': 0.9103641456582633,
  'LightGBMLarge': 0.8655462184873949,
  'WeightedEnsemble_L2': 0.969187675070028},
 'model_best': 'WeightedEnsemble_L2',
 'model_paths': {'LightGBMXT': ['LightGBMX

In [26]:
y_pred_iou_autg = autoglue_predictor.predict(X_test)
record_classification_results(model_results, 'IoU', 'Autogluon_Tab', y_test_cls, y_pred_iou_autg)


Test ROC-AUC 0.4988
Test Accuracy 0.9901
Test F1 0.0000
Test Kappa -0.0037


{'accuracy': 0.9900990099009901,
 'precision': 0.0,
 'recall': 0.0,
 'f1': 0.0,
 'roc_auc': 0.49875311720698257,
 'kappa': -0.003726708074534013}

##### Images

In [27]:
X_train_img = pd.merge(X_train, img_path, left_index=True, right_index=True)
X_test_img = pd.merge(X_test, img_path, how="left", right_index=True, left_index=True)
train_img = pd.merge(X_train_img, y_train_cls, left_index=True, right_index=True)

In [28]:
#autoglue_model_img = MultiModalPredictor(label="iou", problem_type="binary", eval_metric="roc_auc", path="../models/assessors/autoglue_tab/iou/img/")
hyperparams = {
    "env.strategy": "ddp_notebook",
    "env.num_gpus": 1,
    "env.num_workers": 0,
    #"optim.max_epochs": 15,
    #"optim.patience": 3,
    #"optim.learning_rate": 1e-4,
    #"optim.weight_decay": 1e-4,
    #"optim.warmup_steps": 500,
    #"env.per_gpu_batch_size": 8,
    #"env.batch_size": 128,
    #"model.timm_image.checkpoint_name": "convnext_base",
}
#autoglue_predictor_img = autoglue_model_img.fit(train_img, hyperparameters=hyperparams)

In [29]:
#autoglue_predictor_img.fit_summary(verbosity=2)

In [30]:
#y_pred = autoglue_predictor_img.predict(X_test_img)
#record_classification_results(model_results, 'IoU', 'Autogluon_Mult', y_test_cls, y_pred)


### LRP


#### Baseline

Predict Only the Mean


In [31]:
mean_lrp_train = np.mean(y_train_lrp_cls)
lrp_pred_test = np.full_like(y_test_lrp_cls, int(mean_lrp_train >= 0.5))
record_classification_results(model_results, 'LRP', 'Constant Mean Predictor', y_test_lrp_cls, lrp_pred_test)


Test ROC-AUC 0.5000
Test Accuracy 0.9975
Test F1 0.0000
Test Kappa 0.0000


{'accuracy': 0.9975247524752475,
 'precision': 0.0,
 'recall': 0.0,
 'f1': 0.0,
 'roc_auc': 0.5,
 'kappa': 0.0}

#### Linear Reg on Conf

In [32]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['conf']),
    ],
    remainder='passthrough'
)
linear_reg_conf_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression())
])
linear_reg_conf_lrp_grid = GridSearchCV(
    linear_reg_conf_lrp_pipeline,
    param_grid=LINEAR_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
linear_reg_conf_lrp_grid.fit(conf_train, y_train_lrp_cls)

best_idx = linear_reg_conf_lrp_grid.best_index_
mean_train_roc_auc = linear_reg_conf_lrp_grid.cv_results_['mean_train_roc_auc'][best_idx]
mean_test_roc_auc = linear_reg_conf_lrp_grid.cv_results_['mean_test_roc_auc'][best_idx]
mean_train_accuracy = -linear_reg_conf_lrp_grid.cv_results_['mean_train_accuracy'][best_idx]
mean_test_accuracy = -linear_reg_conf_lrp_grid.cv_results_['mean_test_accuracy'][best_idx]
mean_train_f1 = -linear_reg_conf_lrp_grid.cv_results_['mean_train_f1'][best_idx]
mean_test_f1 = -linear_reg_conf_lrp_grid.cv_results_['mean_test_f1'][best_idx]
print(f"Mean CV train ROC-AUC {mean_train_roc_auc:.4f}")
print(f"Mean CV test ROC-AUC {mean_test_roc_auc:.4f}")
print(f"Mean CV train Accuracy {mean_train_accuracy:.4f}")
print(f"Mean CV test Accuracy {mean_test_accuracy:.4f}")
print(f"Mean CV train F1 {mean_train_f1:.4f}")
print(f"Mean CV test F1 {mean_test_f1:.4f}")
print(f"Best params: {linear_reg_conf_lrp_grid.best_params_}")
best_linear_reg_conf_lrp = linear_reg_conf_lrp_grid.best_estimator_


/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=10.
  warnings.warn(
/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in th

Mean CV train ROC-AUC 0.9818
Mean CV test ROC-AUC nan
Mean CV train Accuracy -0.9950
Mean CV test Accuracy -0.9951
Mean CV train F1 -0.0000
Mean CV test F1 -0.0000
Best params: {'model__C': 0.1, 'model__max_iter': 1000, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}


/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/model_selection/_search.py:1135: UserWarning: One or more of the test scores are non-finite: [nan nan nan]
  warnings.warn(


In [33]:
y_pred_lrp_baseline = best_linear_reg_conf_lrp.predict(conf_test)
record_classification_results(model_results, 'LRP', 'Univariate Linear Regression (Confidence)', y_test_lrp_cls, y_pred_lrp_baseline)


Test ROC-AUC 0.5000
Test Accuracy 0.9975
Test F1 0.0000
Test Kappa 0.0000


{'accuracy': 0.9975247524752475,
 'precision': 0.0,
 'recall': 0.0,
 'f1': 0.0,
 'roc_auc': 0.5,
 'kappa': 0.0}

#### Linear Regression


Train models with cv and then test.


In [34]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
linear_reg_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression())
])
linear_reg_lrp_grid = GridSearchCV(
    linear_reg_lrp_pipeline,
    param_grid=LINEAR_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
linear_reg_lrp_grid.fit(X_train, y_train_lrp_cls)

best_idx = linear_reg_lrp_grid.best_index_
mean_train_roc_auc = linear_reg_lrp_grid.cv_results_['mean_train_roc_auc'][best_idx]
mean_test_roc_auc = linear_reg_lrp_grid.cv_results_['mean_test_roc_auc'][best_idx]
mean_train_accuracy = -linear_reg_lrp_grid.cv_results_['mean_train_accuracy'][best_idx]
mean_test_accuracy = -linear_reg_lrp_grid.cv_results_['mean_test_accuracy'][best_idx]
mean_train_f1 = -linear_reg_lrp_grid.cv_results_['mean_train_f1'][best_idx]
mean_test_f1 = -linear_reg_lrp_grid.cv_results_['mean_test_f1'][best_idx]
print(f"Mean CV train ROC-AUC {mean_train_roc_auc:.4f}")
print(f"Mean CV test ROC-AUC {mean_test_roc_auc:.4f}")
print(f"Mean CV train Accuracy {mean_train_accuracy:.4f}")
print(f"Mean CV test Accuracy {mean_test_accuracy:.4f}")
print(f"Mean CV train F1 {mean_train_f1:.4f}")
print(f"Mean CV test F1 {mean_test_f1:.4f}")
print(f"Best params: {linear_reg_lrp_grid.best_params_}")
best_linear_reg_lrp = linear_reg_lrp_grid.best_estimator_


/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=10.
  warnings.warn(
/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sk

Mean CV train ROC-AUC 0.9925
Mean CV test ROC-AUC nan
Mean CV train Accuracy -0.9950
Mean CV test Accuracy -0.9951
Mean CV train F1 -0.0000
Mean CV test F1 -0.0000
Best params: {'model__C': 0.1, 'model__max_iter': 1000, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}


/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/model_selection/_search.py:1135: UserWarning: One or more of the test scores are non-finite: [nan nan nan]
  warnings.warn(


In [35]:
y_pred_lrp_lr = best_linear_reg_lrp.predict(X_test)
record_classification_results(model_results, 'LRP', 'Linear Regression', y_test_lrp_cls, y_pred_lrp_lr)


Test ROC-AUC 0.5000
Test Accuracy 0.9975
Test F1 0.0000
Test Kappa 0.0000


{'accuracy': 0.9975247524752475,
 'precision': 0.0,
 'recall': 0.0,
 'f1': 0.0,
 'roc_auc': 0.5,
 'kappa': 0.0}

#### Decision Trees


In [36]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
decision_tree_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeClassifier(random_state=SEED))
])
decision_tree_lrp_grid = GridSearchCV(
    decision_tree_lrp_pipeline,
    param_grid=TREE_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
decision_tree_lrp_grid.fit(X_train, y_train_lrp_cls)

best_idx = decision_tree_lrp_grid.best_index_
mean_train_roc_auc = decision_tree_lrp_grid.cv_results_['mean_train_roc_auc'][best_idx]
mean_test_roc_auc = decision_tree_lrp_grid.cv_results_['mean_test_roc_auc'][best_idx]
mean_train_accuracy = -decision_tree_lrp_grid.cv_results_['mean_train_accuracy'][best_idx]
mean_test_accuracy = -decision_tree_lrp_grid.cv_results_['mean_test_accuracy'][best_idx]
mean_train_f1 = -decision_tree_lrp_grid.cv_results_['mean_train_f1'][best_idx]
mean_test_f1 = -decision_tree_lrp_grid.cv_results_['mean_test_f1'][best_idx]
print(f"Mean CV train ROC-AUC {mean_train_roc_auc:.4f}")
print(f"Mean CV test ROC-AUC {mean_test_roc_auc:.4f}")
print(f"Mean CV train Accuracy {mean_train_accuracy:.4f}")
print(f"Mean CV test Accuracy {mean_test_accuracy:.4f}")
print(f"Mean CV train F1 {mean_train_f1:.4f}")
print(f"Mean CV test F1 {mean_test_f1:.4f}")
print(f"Best params: {decision_tree_lrp_grid.best_params_}")
best_decision_tree_lrp = decision_tree_lrp_grid.best_estimator_


/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=10.
  warnings.warn(
/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in th

Mean CV train ROC-AUC 0.9986
Mean CV test ROC-AUC nan
Mean CV train Accuracy -0.9967
Mean CV test Accuracy -0.9918
Mean CV train F1 -0.5600
Mean CV test F1 -0.0000
Best params: {'model__max_depth': None, 'model__max_features': None, 'model__min_samples_leaf': 2, 'model__min_samples_split': 2}


/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/metrics/_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/felix/.cache/pypoetry/vir

In [37]:
y_pred_lrp_dt = best_decision_tree_lrp.predict(X_test)
record_classification_results(model_results, 'LRP', 'Decision Tree', y_test_lrp_cls, y_pred_lrp_dt)


Test ROC-AUC 0.4963
Test Accuracy 0.9901
Test F1 0.0000
Test Kappa -0.0037


{'accuracy': 0.9900990099009901,
 'precision': 0.0,
 'recall': 0.0,
 'f1': 0.0,
 'roc_auc': 0.49627791563275436,
 'kappa': -0.003726708074534013}

#### Random Forest


In [38]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
rand_forest_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(random_state=SEED, n_jobs=-1))
])
rand_forest_lrp_grid = GridSearchCV(
    rand_forest_lrp_pipeline,
    param_grid=RF_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
rand_forest_lrp_grid.fit(X_train, y_train_lrp_cls)

best_idx = rand_forest_lrp_grid.best_index_
mean_train_roc_auc = rand_forest_lrp_grid.cv_results_['mean_train_roc_auc'][best_idx]
mean_test_roc_auc = rand_forest_lrp_grid.cv_results_['mean_test_roc_auc'][best_idx]
mean_train_accuracy = -rand_forest_lrp_grid.cv_results_['mean_train_accuracy'][best_idx]
mean_test_accuracy = -rand_forest_lrp_grid.cv_results_['mean_test_accuracy'][best_idx]
mean_train_f1 = -rand_forest_lrp_grid.cv_results_['mean_train_f1'][best_idx]
mean_test_f1 = -rand_forest_lrp_grid.cv_results_['mean_test_f1'][best_idx]
print(f"Mean CV train ROC-AUC {mean_train_roc_auc:.4f}")
print(f"Mean CV test ROC-AUC {mean_test_roc_auc:.4f}")
print(f"Mean CV train Accuracy {mean_train_accuracy:.4f}")
print(f"Mean CV test Accuracy {mean_test_accuracy:.4f}")
print(f"Mean CV train F1 {mean_train_f1:.4f}")
print(f"Mean CV test F1 {mean_test_f1:.4f}")
print(f"Best params: {rand_forest_lrp_grid.best_params_}")
best_rand_forest_lrp = rand_forest_lrp_grid.best_estimator_


/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=10.
  warnings.warn(


/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in 

Mean CV train ROC-AUC 0.9987
Mean CV test ROC-AUC nan
Mean CV train Accuracy -0.9967
Mean CV test Accuracy -0.9934
Mean CV train F1 -0.5600
Mean CV test F1 -0.0000
Best params: {'model__max_depth': None, 'model__max_features': None, 'model__min_samples_leaf': 2, 'model__min_samples_split': 2, 'model__n_estimators': 100}


/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/model_selection/_search.py:1135: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(


In [39]:
y_pred_lrp_rf = best_rand_forest_lrp.predict(X_test)
record_classification_results(model_results, 'LRP', 'Random Forest', y_test_lrp_cls, y_pred_lrp_rf)


Test ROC-AUC 0.4963
Test Accuracy 0.9901
Test F1 0.0000
Test Kappa -0.0037


{'accuracy': 0.9900990099009901,
 'precision': 0.0,
 'recall': 0.0,
 'f1': 0.0,
 'roc_auc': 0.49627791563275436,
 'kappa': -0.003726708074534013}

#### MLP


In [40]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
mlp_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', MLPClassifier(max_iter=500, random_state=SEED))
])
mlp_lrp_grid = GridSearchCV(
    mlp_lrp_pipeline,
    param_grid=MLP_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
mlp_lrp_grid.fit(X_train, y_train_lrp_cls)

best_idx = mlp_lrp_grid.best_index_
mean_train_roc_auc = mlp_lrp_grid.cv_results_['mean_train_roc_auc'][best_idx]
mean_test_roc_auc = mlp_lrp_grid.cv_results_['mean_test_roc_auc'][best_idx]
mean_train_accuracy = -mlp_lrp_grid.cv_results_['mean_train_accuracy'][best_idx]
mean_test_accuracy = -mlp_lrp_grid.cv_results_['mean_test_accuracy'][best_idx]
mean_train_f1 = -mlp_lrp_grid.cv_results_['mean_train_f1'][best_idx]
mean_test_f1 = -mlp_lrp_grid.cv_results_['mean_test_f1'][best_idx]
print(f"Mean CV train ROC-AUC {mean_train_roc_auc:.4f}")
print(f"Mean CV test ROC-AUC {mean_test_roc_auc:.4f}")
print(f"Mean CV train Accuracy {mean_train_accuracy:.4f}")
print(f"Mean CV test Accuracy {mean_test_accuracy:.4f}")
print(f"Mean CV train F1 {mean_train_f1:.4f}")
print(f"Mean CV test F1 {mean_test_f1:.4f}")
print(f"Best params: {mlp_lrp_grid.best_params_}")
best_mlp_lrp = mlp_lrp_grid.best_estimator_


/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=10.
  warnings.warn(
/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sk

Mean CV train ROC-AUC 1.0000
Mean CV test ROC-AUC nan
Mean CV train Accuracy -0.9963
Mean CV test Accuracy -0.9951
Mean CV train F1 -0.3000
Mean CV test F1 -0.0000
Best params: {'model__activation': 'relu', 'model__alpha': 0.0001, 'model__hidden_layer_sizes': (100,), 'model__learning_rate_init': 0.001}


In [41]:
y_pred_lrp_mlp = best_mlp_lrp.predict(X_test)
record_classification_results(model_results, 'LRP', 'MLP', y_test_lrp_cls, y_pred_lrp_mlp)


Test ROC-AUC 0.5000
Test Accuracy 0.9975
Test F1 0.0000
Test Kappa 0.0000


{'accuracy': 0.9975247524752475,
 'precision': 0.0,
 'recall': 0.0,
 'f1': 0.0,
 'roc_auc': 0.5,
 'kappa': 0.0}

#### XGBoost

In [42]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
xgb_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb.XGBClassifier(eval_metric="logloss"))
])
xgb_lrp_grid = GridSearchCV(
    xgb_lrp_pipeline,
    param_grid=XGB_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
xgb_lrp_grid.fit(X_train, y_train_lrp_cls)


best_idx = xgb_lrp_grid.best_index_
mean_train_roc_auc = xgb_lrp_grid.cv_results_['mean_train_roc_auc'][best_idx]
mean_test_roc_auc = xgb_lrp_grid.cv_results_['mean_test_roc_auc'][best_idx]
mean_train_accuracy = -xgb_lrp_grid.cv_results_['mean_train_accuracy'][best_idx]
mean_test_accuracy = -xgb_lrp_grid.cv_results_['mean_test_accuracy'][best_idx]
mean_train_f1 = -xgb_lrp_grid.cv_results_['mean_train_f1'][best_idx]
mean_test_f1 = -xgb_lrp_grid.cv_results_['mean_test_f1'][best_idx]
print(f"Mean CV train ROC-AUC {mean_train_roc_auc:.4f}")
print(f"Mean CV test ROC-AUC {mean_test_roc_auc:.4f}")
print(f"Mean CV train Accuracy {mean_train_accuracy:.4f}")
print(f"Mean CV test Accuracy {mean_test_accuracy:.4f}")
print(f"Mean CV train F1 {mean_train_f1:.4f}")
print(f"Mean CV test F1 {mean_test_f1:.4f}")
print(f"Best params: {xgb_lrp_grid.best_params_}")
best_xgb_lrp = xgb_lrp_grid.best_estimator_

/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=10.
  warnings.warn(
/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/sk

Mean CV train ROC-AUC 0.9912
Mean CV test ROC-AUC nan
Mean CV train Accuracy -0.9950
Mean CV test Accuracy -0.9951
Mean CV train F1 -0.0000
Mean CV test F1 -0.0000
Best params: {'model__learning_rate': 0.01, 'model__n_estimators': 50}


In [43]:
y_pred_lrp_xgb = best_xgb_lrp.predict(X_test)
record_classification_results(model_results, 'LRP', 'XGBoost', y_test_lrp_cls, y_pred_lrp_xgb)


Test ROC-AUC 0.5000
Test Accuracy 0.9975
Test F1 0.0000
Test Kappa 0.0000


{'accuracy': 0.9975247524752475,
 'precision': 0.0,
 'recall': 0.0,
 'f1': 0.0,
 'roc_auc': 0.5,
 'kappa': 0.0}

#### Autogluon

##### Tabluar

In [44]:
train_lrp = pd.merge(X_train, y_train_lrp_cls, left_index=True, right_index=True)

In [45]:
autoglue_model_lrp = TabularPredictor(label="lrp", problem_type="binary", eval_metric="roc_auc", path="../models/assessors/autoglue_tab/lrp/tab/")
autoglue_predictor_lrp = autoglue_model_lrp.fit(train_lrp)


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.14
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Debian 6.12.48-1 (2025-09-20)
CPU Count:          32
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 23.56/23.56 GB
Total GPU Memory:   Free: 23.56 GB, Allocated: 0.00 GB, Total: 23.56 GB
GPU Count:          1
Memory Avail:       1.37 GB / 30.47 GB (4.5%)
Disk Space Avail:   1389.81 GB / 3542.28 GB (39.2%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : New in v1.5: The state-of-the-art for tabular data. Massively better than 'best' on datasets <100000 samples by using new Tabular F

In [46]:
autoglue_predictor_lrp.fit_summary(verbosity=2)

*** Summary of fit() ***
Estimated performance of each model:
                  model  score_val eval_metric  pred_time_val  fit_time  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0              CatBoost   1.000000     roc_auc       0.004517  1.287565                0.004517           1.287565            1       True          5
1   WeightedEnsemble_L2   1.000000     roc_auc       0.004730  1.301745                0.000213           0.014179            2       True         12
2       NeuralNetFastAI   1.000000     roc_auc       0.006337  0.347354                0.006337           0.347354            1       True          8
3              LightGBM   0.991736     roc_auc       0.002133  0.241744                0.002133           0.241744            1       True          2
4      RandomForestGini   0.991736     roc_auc       0.015447  0.236237                0.015447           0.236237            1       True          3
5      RandomForestEntr   0.991736    

/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-NSMX3dx--py3.11/lib/python3.11/site-packages/autogluon/core/utils/plots.py:169: UserWarning: AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"
  warnings.warn('AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"')


{'model_types': {'LightGBMXT': 'LGBModel',
  'LightGBM': 'LGBModel',
  'RandomForestGini': 'RFModel',
  'RandomForestEntr': 'RFModel',
  'CatBoost': 'CatBoostModel',
  'ExtraTreesGini': 'XTModel',
  'ExtraTreesEntr': 'XTModel',
  'NeuralNetFastAI': 'NNFastAiTabularModel',
  'XGBoost': 'XGBoostModel',
  'NeuralNetTorch': 'TabularNeuralNetTorchModel',
  'LightGBMLarge': 'LGBModel',
  'WeightedEnsemble_L2': 'WeightedEnsembleModel'},
 'model_performance': {'LightGBMXT': 0.9834710743801653,
  'LightGBM': 0.9917355371900827,
  'RandomForestGini': 0.9917355371900827,
  'RandomForestEntr': 0.9917355371900826,
  'CatBoost': 1.0,
  'ExtraTreesGini': 0.9834710743801653,
  'ExtraTreesEntr': 0.9834710743801653,
  'NeuralNetFastAI': 1.0,
  'XGBoost': 0.975206611570248,
  'NeuralNetTorch': 0.9752066115702479,
  'LightGBMLarge': 0.9834710743801652,
  'WeightedEnsemble_L2': 1.0},
 'model_best': 'WeightedEnsemble_L2',
 'model_paths': {'LightGBMXT': ['LightGBMXT'],
  'LightGBM': ['LightGBM'],
  'RandomFo

In [47]:
y_pred_lrp_autg = autoglue_predictor_lrp.predict(X_test)
record_classification_results(model_results, 'LRP', 'Autogluon_Tab', y_test_lrp_cls, y_pred_lrp_autg)


Test ROC-AUC 0.4988
Test Accuracy 0.9950
Test F1 0.0000
Test Kappa -0.0025


{'accuracy': 0.995049504950495,
 'precision': 0.0,
 'recall': 0.0,
 'f1': 0.0,
 'roc_auc': 0.4987593052109181,
 'kappa': -0.002481389578163684}

##### Images

In [48]:
X_train_img = pd.merge(X_train, img_path, left_index=True, right_index=True)
X_test_img = pd.merge(X_test, img_path, how="left", right_index=True, left_index=True)
train_lrp_img = pd.merge(X_train_img, y_train_lrp_cls, left_index=True, right_index=True)

In [49]:
#autoglue_model_lrp_img = MultiModalPredictor(label="lrp", problem_type="binary", eval_metric="roc_auc", path="../models/assessors/autoglue_tab/lrp/img/")
hyperparams = {
    "env.strategy": "ddp_notebook",
    "env.num_gpus": 1,
    "env.num_workers": 0,
    #"optim.max_epochs": 15,
    #"optim.patience": 3,
    #"optim.learning_rate": 1e-4,
    #"optim.weight_decay": 1e-4,
    #"optim.warmup_steps": 500,
    #"env.per_gpu_batch_size": 8,
    #"env.batch_size": 128,
    #"model.timm_image.checkpoint_name": "convnext_base",
}
#autoglue_predictor_lrp_img = autoglue_model_lrp_img.fit(train_lrp_img, hyperparameters=hyperparams)

In [50]:
#autoglue_predictor_lrp_img.fit_summary(verbosity=2)

In [51]:
#y_pred = autoglue_predictor_lrp_img.predict(X_test_img)
#record_classification_results(model_results, 'LRP', 'Autogluon_Mult', y_test_lrp_cls, y_pred)


### Save predictions

In [52]:
test_inst = X_test.copy()
metrics = ["iou", "lrp"]
models = ["baseline", "lr", "dt", "rf", "mlp", "xgb", "autg"]
for metric in metrics:
    for model in models:
        if metric == "iou":
            test_inst["GT"] = y_test_cls
        else:
            test_inst["GT"] = y_test_lrp_cls
        var_name = f"y_pred_{metric}_{model}"
        test_inst[model] = globals()[var_name]
    #test_inst.to_csv(f"../results/assessors/{metric}_test_ass_" + data_path_file)


# Final Model Comparison


In [53]:
results_df = pd.DataFrame(model_results, index=None)
results_df["test_roc_auc"] = results_df["test_roc_auc"].round(4)
results_df["test_accuracy"] = results_df["test_accuracy"].round(4)
results_df["test_f1"] = results_df["test_f1"].round(4)
results_df["test_kappa"] = results_df["test_kappa"].round(4)
display(results_df)


,target,model,test_roc_auc,test_accuracy,test_f1,test_kappa
0,IoU,Constant Mean Predictor,0.5000,0.9926,0.0,0.0000
1,IoU,Univariate Linear Regression (Confidence),0.4963,0.9851,0.0,-0.0075
2,IoU,Linear Regression,0.5000,0.9926,0.0,0.0000
3,IoU,Decision Tree,0.5000,0.9926,0.0,0.0000
4,IoU,Random Forest,0.4975,0.9876,0.0,-0.0060
5,IoU,MLP,0.6592,0.9802,0.2,0.1916
6,IoU,XGBoost,0.4963,0.9851,0.0,-0.0075
7,IoU,Autogluon_Tab,0.4988,0.9901,0.0,-0.0037
8,LRP,Constant Mean Predictor,0.5000,0.9975,0.0,0.0000
9,LRP,Univariate Linear Regression (Confidence),0.5000,0.9975,0.0,0.0000


In [54]:
print(results_df.to_latex(index=False))

\begin{tabular}{llrrrr}
\toprule
target & model & test_roc_auc & test_accuracy & test_f1 & test_kappa \\
\midrule
IoU & Constant Mean Predictor & 0.500000 & 0.992600 & 0.000000 & 0.000000 \\
IoU & Univariate Linear Regression (Confidence) & 0.496300 & 0.985100 & 0.000000 & -0.007500 \\
IoU & Linear Regression & 0.500000 & 0.992600 & 0.000000 & 0.000000 \\
IoU & Decision Tree & 0.500000 & 0.992600 & 0.000000 & 0.000000 \\
IoU & Random Forest & 0.497500 & 0.987600 & 0.000000 & -0.006000 \\
IoU & MLP & 0.659200 & 0.980200 & 0.200000 & 0.191600 \\
IoU & XGBoost & 0.496300 & 0.985100 & 0.000000 & -0.007500 \\
IoU & Autogluon_Tab & 0.498800 & 0.990100 & 0.000000 & -0.003700 \\
LRP & Constant Mean Predictor & 0.500000 & 0.997500 & 0.000000 & 0.000000 \\
LRP & Univariate Linear Regression (Confidence) & 0.500000 & 0.997500 & 0.000000 & 0.000000 \\
LRP & Linear Regression & 0.500000 & 0.997500 & 0.000000 & 0.000000 \\
LRP & Decision Tree & 0.496300 & 0.990100 & 0.000000 & -0.003700 \\
LRP & Ran